In [ ]:
!pip install yfinance
!pip install ta

  Preparing metadata (setup.py) ... done
  Created wheel for ta: filename=ta-0.11.0-py3-none-any.whl size=29412 sha256=eda8aad5abeb098402f2380730da95d097c9f63e9c1c48c7f5c6ace0208a16b8
  Stored in directory: /root/.cache/pip/wheels/5f/67/4f/8a9f252836e053e532c6587a3230bc72a4deb16b03a829610b
Successfully built ta


In [ ]:
import pandas as pd
import yfinance as yf
import requests
from datetime import datetime
from dateutil.relativedelta import relativedelta

# Data Retrieval

Let's obtain all NasDaq30 data

In [ ]:
dow30_tickers = [
    'AAPL',  # Apple Inc.
    'MSFT',  # Microsoft Corp.
    'JNJ',   # Johnson & Johnson
    'V',     # Visa Inc.
    'PG',    # Procter & Gamble Co.
    'XOM',   # ExxonMobil Corp.
    'UNH',   # UnitedHealth Group
    'NVDA',  # NVIDIA Corp.
    'GS',    # Goldman Sachs
    'HD',    # The Home Depot
    'MCD',   # McDonald's Corp.
    'AMGN',  # Amgen Inc.
    'VZ',    # Verizon Communications
    'KO',    # The Coca-Cola Company
    'IBM',   # International Business Machines
    'CAT',   # Caterpillar Inc.
    'MRK',   # Merck & Co.
    'TRV',   # The Travelers Companies
    'BA',    # Boeing
    'NKE',   # Nike Inc.
    'WMT',   # Walmart Inc.
    'JPM',   # JPMorgan Chase & Co.
    'DIS',   # Walt Disney Co.
    'AXP',   # American Express
    'MMM',   # 3M Company
    'DOW',   # Dow Inc.
    'CRM',   # Salesforce
    'INTC',  # Intel Corp.
    'HON'    # Honeywell International Inc.
]


In [ ]:
def hist_data(symbols):

  # Initialize an empty list to store DataFrames for each symbol
  dfs = {}

  for i in range(1, len(symbols)+1):

      print(symbols[i-1])

      '''
      url = f'https://eodhd.com/api/eod/{symbols[i-1]}.NSE?from={start}&to={end}&api_token=5ffde981a31603.50022638&fmt=json' # with free API token, I can only do over the past year
      nasdaq = requests.get(url).json()
      '''

      stock = yf.Ticker(symbols[i-1])
      historical_data = stock.history(period='5y')  # Data for the past 5 years

      # Convert data to DataFrame
      dfs[f'df{i}'] = pd.DataFrame(historical_data)

      # changing the date to the index column
      #dfs[f'df{i}'] = dfs[f'df{i}'].set_index('date')

      # renaming the 'close' column to the name of the symbol
      #dfs[f'df{i}'] = dfs[f'df{i}'].rename(columns={'close': symbols[i-1]})

      # adding returns
      dfs[f'df{i}']['Returns'] = (dfs[f'df{i}']['Close'] / dfs[f'df{i}']['Close'].shift(1)) - 1

      # dropping rows with NaN values
      dfs[f'df{i}'] = dfs[f'df{i}'].dropna()

      # using only specific coloumns
      dfs[f'df{i}'] = dfs[f'df{i}'][['Open', 'High', 'Low', 'Close', 'Volume', 'Returns']] # not adding returns here

      print(dfs[f'df{i}'])

  return dfs


In [ ]:
def get_returns(symbols):

    # Dictionary to hold returns for each company
    returns_dict = {}

    for i in range(1, len(symbols)+1):

      '''
      url = f'https://eodhd.com/api/eod/{symbols[i-1]}.NSE?from={start}&to={end}&api_token=5ffde981a31603.50022638&fmt=json' # with free API token, I can only do over the past year
      nasdaq = requests.get(url).json()
      '''

      stock = yf.Ticker(symbols[i-1])
      historical_data = stock.history(period='5y')  #setting it to 5

      # adding returns
      historical_data['Returns'] = (historical_data['Close']/ historical_data['Close'].shift(1)) - 1

      # drop NAN
      historical_data = historical_data.dropna()

      returns_dict[symbols[i-1]] = historical_data['Returns']

    # Create a DataFrame from the returns dictionary
    returns_df = pd.DataFrame(returns_dict)

    # Drop the first row with NaN values due to pct_change
    returns_df = returns_df.dropna()

    return returns_df


In [ ]:
#gathering everything
data_dfs = hist_data(dow30_tickers)
return_data = get_returns(dow30_tickers)

AAPL
                                 Open        High         Low       Close  \
Date                                                                        
2019-11-12 00:00:00-05:00   63.392040   63.692585   63.239353   63.491413   
2019-11-13 00:00:00-05:00   63.290237   64.174889   63.275696   64.099754   
2019-11-14 00:00:00-05:00   63.925251   64.199131   63.525341   63.656223   
2019-11-15 00:00:00-05:00   63.908287   64.417267   63.745903   64.412422   
2019-11-18 00:00:00-05:00   64.422119   64.817185   64.041603   64.737206   
...                               ...         ...         ...         ...   
2024-11-04 00:00:00-05:00  220.747138  222.545148  219.468546  221.766006   
2024-11-05 00:00:00-05:00  221.556241  223.703872  220.896963  223.204422   
2024-11-06 00:00:00-05:00  222.365355  225.821559  220.946918  222.475235   
2024-11-07 00:00:00-05:00  224.383137  227.629565  224.323205  227.229996   
2024-11-08 00:00:00-05:00  227.169998  228.660004  226.410004  226.9600

In [ ]:
return_data

,AAPL,MSFT,JNJ,V,PG,XOM,UNH,NVDA,GS,HD,...,NKE,WMT,JPM,DIS,AXP,MMM,DOW,CRM,INTC,HON
Date,,,,,,,,,,,,,,,,,,,,,
2019-11-12 00:00:00-05:00,-0.000915,0.006570,-0.004773,0.001114,-0.000503,-0.013790,0.004565,0.006869,0.006301,0.005695,...,-0.006107,0.000672,-0.005850,0.013456,-0.000083,0.004292,-0.011647,0.006676,-0.002571,0.006525
2019-11-13 00:00:00-05:00,0.009581,0.001632,-0.000609,-0.001836,0.011571,-0.008217,-0.006582,-0.004961,-0.004945,0.007207,...,0.020000,0.015615,-0.005265,0.073171,-0.005211,-0.001698,-0.027194,-0.001596,-0.005326,-0.000330
2019-11-14 00:00:00-05:00,-0.006919,0.005091,-0.002361,0.003573,-0.001243,-0.004361,0.008913,0.005849,0.000502,0.007453,...,-0.000219,-0.002728,0.000934,-0.010557,0.005571,0.002228,0.005404,0.002768,-0.001382,-0.007733
2019-11-15 00:00:00-05:00,0.011879,0.012900,0.030391,0.000111,0.000332,0.010073,0.053043,-0.026693,0.003737,0.003128,...,0.019393,-0.014753,0.007231,-0.016853,-0.001406,0.005558,0.016868,0.000981,0.002594,0.011689
2019-11-18 00:00:00-05:00,0.005042,0.002467,-0.000815,-0.000612,0.011200,-0.009684,0.013103,0.039620,-0.002361,0.006574,...,0.012253,0.011609,0.008415,0.020599,-0.009109,-0.007796,-0.009843,-0.002696,0.005004,-0.010729
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-11-04 00:00:00-05:00,-0.004038,-0.004654,-0.011803,0.003818,-0.000121,0.031840,-0.017249,0.004801,-0.015173,0.007591,...,-0.011145,0.003163,-0.014174,-0.000209,-0.009168,-0.010769,-0.020829,0.011163,-0.029310,-0.007888
2024-11-05 00:00:00-05:00,0.006486,0.007345,0.000695,0.004934,0.004119,0.002951,0.016602,0.028372,0.030285,0.011427,...,0.010234,0.014918,0.007781,0.008769,0.021281,0.005324,-0.001043,-0.001745,0.035524,0.005975
2024-11-06 00:00:00-05:00,-0.003267,0.021193,-0.002968,0.048109,-0.028415,0.017149,0.052308,0.040740,0.130978,-0.029293,...,-0.034111,-0.002868,0.115445,0.023388,0.069653,0.058489,0.009395,0.031631,0.074185,0.036498


# Normalise Pricing

In [ ]:
def normalise(df):

    # Create a copy to avoid modifying the original DataFrame
    df_normalized = df.copy()

    # Normalize each price column by dividing by the first value (t=1)
    for column in ['Open', 'High', 'Low', 'Close']:
        df_normalized[column] = df[column] / df[column].iloc[0]
        df_normalized.rename(columns={column: f"{column}_normalized"}, inplace=True)

    # Ensure the volume column is present without any changes
    df_normalized['Volume'] = df['Volume']

    # Display the modified DataFrame
    #print(df_normalized)

    return df_normalized


In [ ]:
def norm_dict(dict):

  norm_dict = {}

  for key, value in dict.items():
    norm_dict[key] = normalise(value)

  return norm_dict

In [ ]:
normalised_dict = norm_dict(data_dfs)
print(normalised_dict) # new line, get rid later

{'df1':                            Open_normalized  High_normalized  Low_normalized  \
Date                                                                          
2019-11-12 00:00:00-05:00         1.000000         1.000000        1.000000   
2019-11-13 00:00:00-05:00         0.998394         1.007572        1.000575   
2019-11-14 00:00:00-05:00         1.008411         1.007953        1.004522   
2019-11-15 00:00:00-05:00         1.008144         1.011378        1.008010   
2019-11-18 00:00:00-05:00         1.016249         1.017657        1.012686   
...                                    ...              ...             ...   
2024-11-04 00:00:00-05:00         3.482253         3.494051        3.470443   
2024-11-05 00:00:00-05:00         3.495017         3.512244        3.493030   
2024-11-06 00:00:00-05:00         3.507780         3.545492        3.493820   
2024-11-07 00:00:00-05:00         3.539611         3.573879        3.547209   
2024-11-08 00:00:00-05:00         3.583573  

# PCA

PC on returns

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
def pca_vectors(data):

    # Remove the date column
    #returned_data = data.drop(columns=['Date'])
    # should we perform PCA on returns also?

    # Perform PCA
    pca = PCA()
    pca.fit(data) #should we standardise the data ?

    principal_components = pca.components_

    return principal_components


In [ ]:
pc_vec = pca_vectors(return_data)

# PC Portfolio

In [ ]:
def portfolio(pc_array, stock_data_list, comp): # comp defines which principal component we are looking at

    # Initialize an empty DataFrame to store the results
    combined_data = None

    # taking PC component as weight
    weights_array = (-1) * pc_array[:][comp-1]

    # now we want to take sum of each column according to the weight assigned to the stock

    # Initialize a dictionary to store the weighted sums for each column type
    weighted_sums = {}

    # Loop through each stock DataFrame and its corresponding weight
    for (tickers, df), weight in zip(stock_data_list.items(), weights_array):
        # Check available columns in the DataFrame
        available_columns = df.columns

        # Iterate through available columns and calculate weighted sum
        for col_type in available_columns:
            # Multiply the selected column by its corresponding weight
            weighted_col = df[col_type] * weight

            # Add to the sum
            if col_type not in weighted_sums:
                weighted_sums[col_type] = weighted_col
            else:
                weighted_sums[col_type] += weighted_col

    # Convert the weighted_sums dictionary to a DataFrame
    result_df = pd.DataFrame(weighted_sums)

    return result_df

In [ ]:
portfolio_data = portfolio(pc_vec, normalised_dict, 1)

In [ ]:
portfolio_data # check if this is correct

,Open_normalized,High_normalized,Low_normalized,Close_normalized,Volume,Returns
Date,,,,,,
2019-11-12 00:00:00-05:00,-5.092427,-5.092427,-5.092427,-5.092427,-1.269951e+08,0.001087
2019-11-13 00:00:00-05:00,-5.063971,-5.084162,-5.072927,-5.100622,-1.414955e+08,-0.008195
2019-11-14 00:00:00-05:00,-5.087365,-5.088635,-5.093422,-5.105355,-2.013992e+08,-0.004939
2019-11-15 00:00:00-05:00,-5.115449,-5.124303,-5.113606,-5.131995,-3.569989e+08,-0.026915
2019-11-18 00:00:00-05:00,-5.118104,-5.132140,-5.122698,-5.145407,-2.147339e+08,-0.013282
...,...,...,...,...,...,...
2024-11-04 00:00:00-05:00,-15.898849,-15.911468,-15.821011,-15.812485,-1.110999e+08,0.025757
2024-11-05 00:00:00-05:00,-15.887702,-16.022629,-15.943846,-16.110072,-9.436692e+07,-0.047356
2024-11-06 00:00:00-05:00,-16.510462,-16.722121,-16.446617,-16.735797,-1.594560e+08,-0.155828


# Indicators

In [ ]:
from ta import add_all_ta_features
from ta.utils import dropna

In [ ]:
def features_ta(weighted_data):

    weighted_data = add_all_ta_features(weighted_data, open="Open_normalized", high="High_normalized", low="Low_normalized", close="Close_normalized", volume="Volume", fillna=True)

    print(weighted_data)

    return weighted_data

In [ ]:
pc_feature_data = features_ta(portfolio_data)

/usr/local/lib/python3.10/dist-packages/ta/trend.py:1030: FutureWarning: Series.__setitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To set a value by position, use `ser.iloc[pos] = value`
  self._psar[i] = high2


                           Open_normalized  High_normalized  Low_normalized  \
Date                                                                          
2019-11-12 00:00:00-05:00        -5.092427        -5.092427       -5.092427   
2019-11-13 00:00:00-05:00        -5.063971        -5.084162       -5.072927   
2019-11-14 00:00:00-05:00        -5.087365        -5.088635       -5.093422   
2019-11-15 00:00:00-05:00        -5.115449        -5.124303       -5.113606   
2019-11-18 00:00:00-05:00        -5.118104        -5.132140       -5.122698   
...                                    ...              ...             ...   
2024-11-04 00:00:00-05:00       -15.898849       -15.911468      -15.821011   
2024-11-05 00:00:00-05:00       -15.887702       -16.022629      -15.943846   
2024-11-06 00:00:00-05:00       -16.510462       -16.722121      -16.446617   
2024-11-07 00:00:00-05:00       -16.766888       -16.840889      -16.767063   
2024-11-08 00:00:00-05:00       -16.892231       -16

/usr/local/lib/python3.10/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [ ]:
pc_feature_data.to_csv('pc_feature_data.csv')

# Correlation of Indicators

we shift the indicators down by 1 as we want correlation between indicators at t-1 and returns at t

In [ ]:
def correlation_matrix(pc_data): # how many components were used

    # DataFrame to store all correlations
    all_correlations_df = pd.DataFrame()

    return_column = pc_data.iloc[:, 5]  # 5th column which contains all the returns
    indicator_columns = pc_data.iloc[:, (6):]  # All columns after the 6th

    # Shifting
    lagged_indicators = indicator_columns.shift(1)

    # Calculate the correlation
    correlation = lagged_indicators.corrwith(return_column)

    # Convert the correlation Series into a DataFrame
    correlation_df = correlation.to_frame(name='Correlation')

    return correlation_df.dropna()

In [ ]:
correlation_matrix_data = correlation_matrix(pc_feature_data)
print(correlation_matrix_data)

                     Correlation
volume_adi             -0.017086
volume_obv              0.023521
volume_cmf              0.016516
volume_fi               0.033353
volume_em              -0.037568
...                          ...
momentum_pvo_signal     0.023681
momentum_pvo_hist      -0.044756
momentum_kama          -0.010008
others_dr               0.148359
others_cr               0.012087

[85 rows x 1 columns]


/usr/local/lib/python3.10/dist-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.10/dist-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


now to filter

In [ ]:
def correlation_filter(corr_mat, threshold):

    # Filter rows where the sum exceeds threshold value
    filtered_df = corr_mat[corr_mat['Correlation'].abs() >= threshold]

    # Extract the names of the rows that were kept
    included_rows = filtered_df.index.tolist()

    return filtered_df, included_rows

In [ ]:
filter_correlation_matrix, indicator_list = correlation_filter(correlation_matrix_data, 0.03)
print(filter_correlation_matrix)
print(indicator_list)

                           Correlation
volume_fi                     0.033353
volume_em                    -0.037568
volume_nvi                    0.039467
volatility_bbhi              -0.066552
volatility_kcw               -0.047224
trend_macd_signal            -0.030660
trend_kst_sig                 0.031894
trend_psar_down_indicator     0.035598
momentum_ppo                  0.035213
momentum_ppo_signal           0.041234
momentum_pvo_hist            -0.044756
others_dr                     0.148359
['volume_fi', 'volume_em', 'volume_nvi', 'volatility_bbhi', 'volatility_kcw', 'trend_macd_signal', 'trend_kst_sig', 'trend_psar_down_indicator', 'momentum_ppo', 'momentum_ppo_signal', 'momentum_pvo_hist', 'others_dr']


In [ ]:
# Sort the rows based on the absolute values in decreasing order
df_sorted = filter_correlation_matrix.reindex(filter_correlation_matrix['Correlation'].abs().sort_values(ascending=False).index)

print(df_sorted)

                           Correlation
others_dr                     0.148359
volatility_bbhi              -0.066552
volatility_kcw               -0.047224
momentum_pvo_hist            -0.044756
momentum_ppo_signal           0.041234
volume_nvi                    0.039467
volume_em                    -0.037568
trend_psar_down_indicator     0.035598
momentum_ppo                  0.035213
volume_fi                     0.033353
trend_kst_sig                 0.031894
trend_macd_signal            -0.030660


# Regression

In [ ]:
!pip install lazypredict

In [ ]:
"""

From the top 10 correlated indicators
We will create a prediction model to predict the returns

"""



"""

A question arises whether it makes sense
To use others_dlr and others_dr as suitable indicators
as they themselves are designed for finding returns

"""


"""

And a final thing to wonder about
Is whether we should use the open, hight, low, etc. data
When predicting, since the indicators are based off of them

"""

'\n\nAnd a final thing to wonder about\nIs whether we should use the open, hight, low, etc. data\nWhen predicting, since the indicators are based off of them\n\n'

In [ ]:
# Extract the first 5 indicators from the index
first_5_names = df_sorted.index[:5]
print(first_5_names)

Index(['others_dr', 'volatility_bbhi', 'volatility_kcw', 'momentum_pvo_hist',
       'momentum_ppo_signal'],
      dtype='object')


In [ ]:
"""

Here we create a dataframe consisting of the
indicators we will build our model around
We will call it 'X_df' to resemble X and Y
Like how we'd generally do while performing regression

"""


# Use the existing time series as the index for the new DataFrame
X_df = pd.DataFrame(index=pc_feature_data.index, columns=first_5_names) # Changed from columns to index

# Let's populate each column with random data for now (replace this with your actual logic)
for name in first_5_names:
    X_df[name] = pc_feature_data[name]  # Changed from .loc to [] to access column data

# Display the new DataFrame
print(X_df)

                           others_dr  volatility_bbhi  volatility_kcw  \
Date                                                                    
2019-11-12 00:00:00-05:00   0.000000              0.0       -0.000000   
2019-11-13 00:00:00-05:00   0.160921              0.0        0.220760   
2019-11-14 00:00:00-05:00   0.092799              0.0        0.084430   
2019-11-15 00:00:00-05:00   0.521805              0.0        0.168114   
2019-11-18 00:00:00-05:00   0.261337              0.0        0.208284   
...                              ...              ...             ...   
2024-11-04 00:00:00-05:00  -0.041423              0.0        1.297188   
2024-11-05 00:00:00-05:00   1.881977              0.0        1.338072   
2024-11-06 00:00:00-05:00   3.884056              0.0        1.427140   
2024-11-07 00:00:00-05:00   0.979323              0.0        1.455581   
2024-11-08 00:00:00-05:00  -0.268880              0.0        1.451097   

                           momentum_pvo_hist  mome

In [ ]:
"""

Doing the same for returns
and storing it as 'Y_df'

"""

Y_df = pc_feature_data['Returns']
print(Y_df)


Date
2019-11-12 00:00:00-05:00    0.001087
2019-11-13 00:00:00-05:00   -0.008195
2019-11-14 00:00:00-05:00   -0.004939
2019-11-15 00:00:00-05:00   -0.026915
2019-11-18 00:00:00-05:00   -0.013282
                               ...   
2024-11-04 00:00:00-05:00    0.025757
2024-11-05 00:00:00-05:00   -0.047356
2024-11-06 00:00:00-05:00   -0.155828
2024-11-07 00:00:00-05:00   -0.018349
2024-11-08 00:00:00-05:00   -0.010824
Name: Returns, Length: 1257, dtype: float64


In [ ]:
"""

Converting the above data to numpy arrays
So that we can carry out regression

"""

X = X_df.to_numpy()
Y = Y_df.to_numpy()

In [ ]:
""""
firslty,
we will manually conduct XGBOOST, LightGBM and Ridge

"""

!pip install xgboost
!pip install lightgbm

import xgboost as xgb
import lightgbm as lgb
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score


/usr/local/lib/python3.10/dist-packages/dask/dataframe/__init__.py:42: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


In [ ]:
# Split the data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42) # gotta figure out the random_state aspect


In [ ]:
"""
XGBoost
"""

xgb_model = xgb.XGBRegressor()
xgb_model.fit(X_train, y_train)

# Make predictions on the test set
y_train_pred = xgb_model.predict(X_train)
in_sample_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
in_sample_r2 = r2_score(y_train, y_train_pred)
print(f"In-sample RMSE: {in_sample_rmse}")
print(f"In-sample R²: {in_sample_r2}")

y_test_pred = xgb_model.predict(X_test)
out_of_sample_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
out_of_sample_r2 = r2_score(y_test, y_test_pred)
print(f"Out-of-sample RMSE: {out_of_sample_rmse}")
print(f"Out-of-sample R²: {out_of_sample_r2}")


In-sample RMSE: 0.003206313871673074
In-sample R²: 0.9981610825093233
Out-of-sample RMSE: 0.04284969022279626
Out-of-sample R²: 0.6175258501405458


In [ ]:
"""
LightGBM
"""

lgb_model = lgb.LGBMRegressor()
lgb_model.fit(X_train, y_train)

# Make predictions on the test set
y_train_pred = lgb_model.predict(X_train)
in_sample_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
in_sample_r2 = r2_score(y_train, y_train_pred)
print(f"In-sample RMSE: {in_sample_rmse}")
print(f"In-sample R²: {in_sample_r2}")

y_test_pred = lgb_model.predict(X_test)
out_of_sample_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
out_of_sample_r2 = r2_score(y_test, y_test_pred)
print(f"Out-of-sample RMSE: {out_of_sample_rmse}")
print(f"Out-of-sample R²: {out_of_sample_r2}")


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001043 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1022
[LightGBM] [Info] Number of data points in the train set: 1005, number of used features: 5
[LightGBM] [Info] Start training from score -0.004465
In-sample RMSE: 0.0250879928688832
In-sample R²: 0.8874146166825668
Out-of-sample RMSE: 0.026672560779019423
Out-of-sample R²: 0.8518041160067662


In [ ]:
"""
RIDGE

"""

ridge_model = Ridge()
ridge_model.fit(X_train, y_train)

# Make predictions on the test set
y_train_pred = ridge_model.predict(X_train)
in_sample_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
in_sample_r2 = r2_score(y_train, y_train_pred)
print(f"In-sample RMSE: {in_sample_rmse}")
print(f"In-sample R²: {in_sample_r2}")

y_test_pred = ridge_model.predict(X_test)
out_of_sample_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
out_of_sample_r2 = r2_score(y_test, y_test_pred)
print(f"Out-of-sample RMSE: {out_of_sample_rmse}")
print(f"Out-of-sample R²: {out_of_sample_r2}")


In-sample RMSE: 0.027284037077828494
In-sample R²: 0.8668419465791809
Out-of-sample RMSE: 0.02869884341672289
Out-of-sample R²: 0.8284323098770006


In [ ]:
from lazypredict.Supervised import LazyRegressor

In [ ]:
# Fit all regression models
reg = LazyRegressor(predictions=True)
models, predictions = reg.fit(X_train, X_test, y_train, y_test)
models

100%|██████████| 42/42 [00:06<00:00,  6.21it/s]


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000119 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1023
[LightGBM] [Info] Number of data points in the train set: 1005, number of used features: 5
[LightGBM] [Info] Start training from score -0.004465


,Adjusted R-Squared,R-Squared,RMSE,Time Taken
Model,,,,
HistGradientBoostingRegressor,0.85,0.85,0.03,0.55
LGBMRegressor,0.84,0.84,0.03,0.18
LassoCV,0.83,0.83,0.03,0.26
LarsCV,0.83,0.83,0.03,0.05
LassoLarsCV,0.83,0.83,0.03,0.07
ElasticNetCV,0.83,0.83,0.03,0.11
OrthogonalMatchingPursuit,0.83,0.83,0.03,0.04
LassoLarsIC,0.83,0.83,0.03,0.02
OrthogonalMatchingPursuitCV,0.83,0.83,0.03,0.04
